## Fase 6C: Construção do Modelo de IA para Previsão de Produtividade (Parte 2A)

Neste notebook, vamos continuar a construção do modelo de IA para previsão de produtividade agrícola, otimizando os modelos treinados na Parte 1.

In [ ]:
# Configuração do ambiente
import sys
sys.path.append('../../')
import setup_notebook
setup_notebook.setup_environment()

%matplotlib inline

## Introdução

Na Parte 1 deste notebook, iniciamos a construção do modelo de IA para previsão de produtividade agrícola. Carregamos as features selecionadas, exploramos e preparamos os dados para modelagem, selecionamos e avaliamos diferentes algoritmos de aprendizado de máquina, e treinamos os modelos com melhor desempenho.

Neste notebook (Parte 2A), vamos focar nas seguintes tarefas:

1. Carregamento dos modelos treinados na Parte 1
2. Otimização dos hiperparâmetros dos melhores modelos
3. Análise inicial dos resultados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
import warnings

# Configurar o estilo dos gráficos
plt.style.use('fivethirtyeight')
sns.set(style="whitegrid")

# Ignorar avisos
warnings.filterwarnings('ignore')

## 1. Carregamento dos Dados e Modelos Treinados

Vamos carregar os dados e os modelos treinados na Parte 1.

In [ ]:
# Carregar os dados
if os.path.exists('../../sprint2/assets/features_relevantes.csv'):
    # Carregar as features relevantes
    df_features = pd.read_csv('../../sprint2/assets/features_relevantes.csv')
    print("Features relevantes carregadas com sucesso!")
else:
    # Se o arquivo não existir, carregar as features finais
    if os.path.exists('../../sprint2/assets/features_finais.csv'):
        df_features = pd.read_csv('../../sprint2/assets/features_finais.csv')
        print("Features finais carregadas com sucesso!")
    else:
        # Se nenhum arquivo existir, carregar os dados originais
        print("Os arquivos de features não foram encontrados. Carregando os dados originais...")
        
        # Carregar os dados de NDVI/EVI
        df_ndvi_nf = pd.read_csv('../../assets/ndvi_mensal_nova_friburgo.csv')
        df_ndvi_t = pd.read_csv('../../assets/ndvi_mensal_teresopolis.csv')
        
        # Carregar os dados de produtividade agrícola
        df_prod_combinados = pd.read_csv('../../assets/dados_produtividade_combinados.csv')
        
        # Filtrar os dados de produtividade para o ano de 2017
        df_prod_2017 = df_prod_combinados[df_prod_combinados['Ano'] == 2017]
        
        # Agrupar os dados de produtividade por município
        df_prod_mean = df_prod_2017.groupby('Município')['Produtividade (t/ha)'].mean().reset_index()
        
        # Filtrar os dados de NDVI/EVI para o ano de 2017
        df_ndvi_nf_2017 = df_ndvi_nf[df_ndvi_nf['Ano'] == 2017]
        df_ndvi_t_2017 = df_ndvi_t[df_ndvi_t['Ano'] == 2017]
        
        # Calcular a média anual do EVI para cada município em 2017
        evi_nf_2017 = df_ndvi_nf_2017['EVI_mean'].mean()
        evi_t_2017 = df_ndvi_t_2017['EVI_mean'].mean()
        
        # Criar um dataframe com os dados de EVI e produtividade
        data = {
            'Município': ['Nova Friburgo', 'Teresópolis'],
            'EVI_mean': [evi_nf_2017, evi_t_2017]
        }
        df_features = pd.DataFrame(data)
        
        # Mesclar com os dados de produtividade
        df_features = pd.merge(df_features, df_prod_mean, on='Município')
        
        print("Dados carregados e processados manualmente.")

# Preparar os dados para modelagem
# Separar as features (X) e o target (y)
X = df_features.drop(columns=['Município', 'Produtividade (t/ha)'])
y = df_features['Produtividade (t/ha)']

# Exibir o dataframe
print("\nDataframe de features:")
print(df_features)

In [ ]:
# Verificar os modelos treinados disponíveis
model_files = [f for f in os.listdir('../../sprint2/models') if f.endswith('_pipeline.pkl')]

if model_files:
    # Carregar os modelos treinados
    trained_models = {}
    for model_file in model_files:
        model_name = model_file.replace('_pipeline.pkl', '').replace('_', ' ').title()
        model_path = os.path.join('../../sprint2/models', model_file)
        trained_models[model_name] = joblib.load(model_path)
    
    print(f"Modelos carregados: {list(trained_models.keys())}")
else:
    # Se não houver modelos treinados, criar modelos básicos
    print("Nenhum modelo treinado encontrado. Criando modelos básicos...")
    
    # Definir os modelos a serem avaliados
    models = {
        'Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(random_state=42)
    }
    
    # Criar pipelines com StandardScaler e os modelos
    trained_models = {}
    for name, model in models.items():
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('model', model)
        ])
        pipeline.fit(X, y)
        trained_models[name] = pipeline
    
    print(f"Modelos criados: {list(trained_models.keys())}")

## 2. Otimização dos Hiperparâmetros dos Melhores Modelos

Vamos otimizar os hiperparâmetros dos melhores modelos para melhorar o desempenho.

In [ ]:
# Definir os espaços de hiperparâmetros para cada modelo
param_grids = {}

# Random Forest
if 'Random Forest' in trained_models:
    param_grids['Random Forest'] = {
        'model__n_estimators': [50, 100, 200],
        'model__max_depth': [None, 5, 10, 15],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4]
    }

# Gradient Boosting
if 'Gradient Boosting' in trained_models:
    param_grids['Gradient Boosting'] = {
        'model__n_estimators': [50, 100, 200],
        'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
        'model__max_depth': [3, 5, 7],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4]
    }

# SVR
if 'Svr' in trained_models:
    param_grids['Svr'] = {
        'model__C': [0.1, 1, 10, 100],
        'model__gamma': [0.01, 0.1, 1, 'scale', 'auto'],
        'model__kernel': ['linear', 'rbf', 'poly']
    }

# Ridge Regression
if 'Ridge Regression' in trained_models:
    param_grids['Ridge Regression'] = {
        'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
    }

# Lasso Regression
if 'Lasso Regression' in trained_models:
    param_grids['Lasso Regression'] = {
        'model__alpha': [0.001, 0.01, 0.1, 1.0, 10.0]
    }

# ElasticNet
if 'Elasticnet' in trained_models:
    param_grids['Elasticnet'] = {
        'model__alpha': [0.001, 0.01, 0.1, 1.0, 10.0],
        'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
    }

In [ ]:
# Otimizar os hiperparâmetros dos modelos
optimized_models = {}
for name, model in trained_models.items():
    if name in param_grids:
        print(f"Otimizando hiperparâmetros para {name}...")
        
        # Usar RandomizedSearchCV para otimização
        grid_search = RandomizedSearchCV(
            model,
            param_distributions=param_grids[name],
            n_iter=10,
            cv=min(5, len(X)),  # Usar no máximo 5-fold CV
            scoring='neg_mean_squared_error',
            random_state=42
        )
        
        # Treinar o modelo com todos os dados
        grid_search.fit(X, y)
        
        # Obter o melhor modelo
        best_model = grid_search.best_estimator_
        
        # Fazer previsões
        y_pred = best_model.predict(X)
        
        # Calcular métricas
        rmse = np.sqrt(mean_squared_error(y, y_pred))
        mae = mean_absolute_error(y, y_pred)
        r2 = r2_score(y, y_pred)
        
        # Armazenar o modelo otimizado e as métricas
        optimized_models[name] = {
            'pipeline': best_model,
            'best_params': grid_search.best_params_,
            'RMSE': rmse,
            'MAE': mae,
            'R²': r2,
            'y_pred': y_pred
        }
        
        print(f"Melhores hiperparâmetros para {name}: {grid_search.best_params_}")
        print(f"RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}\n")
    else:
        # Se não houver grade de hiperparâmetros para o modelo, usar o modelo original
        y_pred = model.predict(X)
        
        # Calcular métricas
        rmse = np.sqrt(mean_squared_error(y, y_pred))
        mae = mean_absolute_error(y, y_pred)
        r2 = r2_score(y, y_pred)
        
        # Armazenar o modelo original e as métricas
        optimized_models[name] = {
            'pipeline': model,
            'best_params': {},
            'RMSE': rmse,
            'MAE': mae,
            'R²': r2,
            'y_pred': y_pred
        }
        
        print(f"{name} (sem otimização): RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}\n")

## 3. Análise Inicial dos Resultados

Vamos analisar inicialmente os resultados dos modelos otimizados.

In [ ]:
# Exibir as métricas dos modelos otimizados
metrics = {name: {'RMSE': model['RMSE'], 'MAE': model['MAE'], 'R²': model['R²']} 
          for name, model in optimized_models.items()}
metrics_df = pd.DataFrame(metrics).T

# Ordenar por RMSE
metrics_df_sorted = metrics_df.sort_values('RMSE')
print("Métricas dos modelos otimizados (ordenados por RMSE):")
print(metrics_df_sorted)

# Identificar o melhor modelo
best_model_name = metrics_df_sorted.index[0]
best_model = optimized_models[best_model_name]
print(f"\nMelhor modelo: {best_model_name}")
print(f"RMSE: {best_model['RMSE']:.4f}, MAE: {best_model['MAE']:.4f}, R²: {best_model['R²']:.4f}")
if best_model['best_params']:
    print(f"Melhores hiperparâmetros: {best_model['best_params']}")

In [ ]:
# Visualizar as métricas dos modelos otimizados
plt.figure(figsize=(12, 6))

# Plotar o RMSE
plt.subplot(1, 2, 1)
metrics_df_sorted['RMSE'].plot(kind='bar')
plt.title('RMSE por Modelo (Otimizado)')
plt.xlabel('Modelo')
plt.ylabel('RMSE')
plt.xticks(rotation=45, ha='right')
plt.grid(True)

# Plotar o R²
plt.subplot(1, 2, 2)
metrics_df_sorted['R²'].plot(kind='bar')
plt.title('R² por Modelo (Otimizado)')
plt.xlabel('Modelo')
plt.ylabel('R²')
plt.xticks(rotation=45, ha='right')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Visualizar as previsões dos modelos otimizados
plt.figure(figsize=(12, 6))

# Plotar os valores reais
plt.scatter(range(len(y)), y, color='blue', label='Valores Reais', s=100)

# Plotar as previsões dos 3 melhores modelos
for i, name in enumerate(metrics_df_sorted.index[:3]):
    plt.scatter(range(len(y)), optimized_models[name]['y_pred'], marker='x', s=100, label=f'Previsões - {name}')

plt.title('Valores Reais vs. Previsões (Modelos Otimizados)')
plt.xlabel('Índice')
plt.ylabel('Produtividade (t/ha)')
plt.xticks(range(len(y)), df_features['Município'])
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 4. Exportação dos Modelos Otimizados

Vamos exportar os modelos otimizados para serem utilizados na próxima parte.

In [ ]:
# Criar diretório para os modelos otimizados
os.makedirs('../../sprint2/optimized_models', exist_ok=True)

# Salvar os modelos otimizados
for name, model in optimized_models.items():
    # Salvar o pipeline (scaler + modelo)
    joblib.dump(model['pipeline'], f'../../sprint2/optimized_models/{name.replace(" ", "_").lower()}_pipeline.pkl')
    
    # Salvar as métricas
    metrics = {
        'RMSE': model['RMSE'],
        'MAE': model['MAE'],
        'R²': model['R²']
    }
    pd.DataFrame([metrics]).to_csv(f'../../sprint2/optimized_models/{name.replace(" ", "_").lower()}_metrics.csv', index=False)
    
    # Salvar os melhores hiperparâmetros
    if model['best_params']:
        pd.DataFrame([model['best_params']]).to_csv(f'../../sprint2/optimized_models/{name.replace(" ", "_").lower()}_best_params.csv', index=False)

# Salvar o nome do melhor modelo
with open('../../sprint2/optimized_models/best_model_name.txt', 'w') as f:
    f.write(best_model_name)

print("Modelos otimizados exportados com sucesso!")

## Conclusão da Parte 2A

Neste notebook, continuamos a construção do modelo de IA para previsão de produtividade agrícola. Carregamos os modelos treinados na Parte 1, otimizamos os hiperparâmetros dos melhores modelos e realizamos uma análise inicial dos resultados.

Principais observações:

1. Otimizamos os hiperparâmetros dos modelos usando RandomizedSearchCV.
2. Identificamos o melhor modelo com base no RMSE: [melhor modelo].
3. Analisamos os resultados dos modelos otimizados e comparamos suas previsões com os valores reais.
4. Exportamos os modelos otimizados para serem utilizados na próxima parte.

Na próxima parte (Fase 6C - Parte 2B), vamos realizar uma análise mais detalhada dos resultados, interpretar o modelo e finalizar o modelo de IA para previsão de produtividade agrícola.